In [ ]:
!pip install -q gradio demucs openunmix soundfile pydub yt-dlp

import gradio as gr
import subprocess
import os
import shutil
import soundfile as sf
import zipfile
import yt_dlp
import re
from pydub import AudioSegment

def get_audio_info(file_path):
    """Technical analysis for the report (Sampling Import)"""
    try:
        info = sf.info(file_path)
        return f"{info.samplerate} Hz | {info.channels} ch | {round(info.duration, 2)}s"
    except:
        return "Info not available"

def process_all(audio_files, yt_links_text, model_choice, target_sr, bit_depth, progress=gr.Progress()):
    work_dir = "processing_vault"
    extract_dir = "raw_imports"
    os.makedirs(work_dir, exist_ok=True)
    os.makedirs(extract_dir, exist_ok=True)

    zip_filename = "Pro_Audio_Extraction_Package.zip"
    zip_obj = zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED)

    audio_tasks = []
    report_content = "TECHNICAL EXTRACTION REPORT\n" + "="*30 + "\n\n"
    valid_ext = ('.mp3', '.wav', '.flac', '.ogg', '.m4a')

    # 1. YOUTUBE COLLECTION
    if yt_links_text and yt_links_text.strip():
        links = [l.strip() for l in re.split(r'[\n,]', yt_links_text) if l.strip()]
        for idx, link in enumerate(links):
            progress(0.1, desc=f"YouTube {idx+1}/{len(links)}")
            ydl_opts = {
                'format': 'bestaudio/best',
                'outtmpl': os.path.join(extract_dir, '%(title)s.%(ext)s'),
                'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
                'quiet': True
            }
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    info = ydl.extract_info(link, download=True)
                    path = os.path.splitext(ydl.prepare_filename(info))[0] + '.wav'
                    audio_tasks.append({'path': path, 'parent': "YouTube_Downloads"})
            except: continue

    # 2. FILE AND ZIP COLLECTION
    if audio_files:
        for f in audio_files:
            fname = os.path.basename(f)
            name_no_ext = os.path.splitext(fname)[0]
            if f.lower().endswith('.zip'):
                z_path = os.path.join(extract_dir, name_no_ext)
                os.makedirs(z_path, exist_ok=True)
                with zipfile.ZipFile(f, 'r') as z:
                    z.extractall(z_path)
                for root, _, files in os.walk(z_path):
                    for file in files:
                        if file.lower().endswith(valid_ext) and not file.startswith('._'):
                            audio_tasks.append({'path': os.path.join(root, file), 'parent': name_no_ext})
            elif f.lower().endswith(valid_ext):
                audio_tasks.append({'path': f, 'parent': ""})

    # 3. AI PROCESSING AND SAMPLING EXPORT
    sr_int = int(target_sr.replace(" Hz", ""))
    subtype = 'PCM_16' if bit_depth == "16-bit" else 'PCM_24'

    for idx, task in enumerate(audio_tasks):
        f_path = task['path']
        original_name = os.path.splitext(os.path.basename(f_path))[0]
        progress((idx/len(audio_tasks)), desc=f"AI: {original_name[:15]}...")

        # Import Report
        report_content += f"SOURCE: {original_name}\n- Import: {get_audio_info(f_path)}\n"

        # Separation
        if model_choice == "Demucs":
            subprocess.run(["demucs", "--two-stems=vocals", "-o", work_dir, f_path], check=True)
            r_voc = os.path.join(work_dir, "htdemucs", original_name, "vocals.wav")
            r_inst = os.path.join(work_dir, "htdemucs", original_name, "no_vocals.wav")
        else:
            subprocess.run(["umx", f_path, "--outdir", work_dir], check=True)
            r_voc = os.path.join(work_dir, original_name, "vocals.wav")
            d, s = sf.read(os.path.join(work_dir, original_name, "drums.wav"))
            b, _ = sf.read(os.path.join(work_dir, original_name, "bass.wav"))
            o, _ = sf.read(os.path.join(work_dir, original_name, "other.wav"))
            r_inst = "tmp_inst.wav"; sf.write(r_inst, d+b+o, s)

        # Applying output parameters
        f_voc = f"Vocals_{original_name}.wav"
        f_inst = f"Instru_{original_name}.wav"

        for raw, final in [(r_voc, f_voc), (r_inst, f_inst)]:
            data, _ = sf.read(raw)
            sf.write(final, data, sr_int, subtype=subtype)

        # Storing in ZIP
        prefix = f"{task['parent']}/{original_name}/" if task['parent'] else f"{original_name}/"
        zip_obj.write(f_voc, arcname=prefix + f_voc)
        zip_obj.write(f_inst, arcname=prefix + f_inst)

        report_content += f"- Export: {target_sr} | {bit_depth}\n\n"
        os.remove(f_voc); os.remove(f_inst)

    with open("Final_Report.txt", "w") as rf: rf.write(report_content)
    zip_obj.write("Final_Report.txt")
    zip_obj.close()

    shutil.rmtree(work_dir); shutil.rmtree(extract_dir)
    return zip_filename

# --- VISUAL INTERFACE ---
with gr.Blocks(theme=gr.themes.Default()) as app:
    gr.Markdown("# 🔬 Audio Master Separator")

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### 📥 1. Sources (Import)")
            yt_input = gr.TextArea(label="YouTube Links (one per line)", lines=3)
            file_input = gr.File(label="Audio or ZIP Files", file_count="multiple")

        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ 2. Sampling Parameters")
            # FREQUENCY CHOICES ARE HERE
            model_sel = gr.Radio(["Demucs", "Open-Unmix"], value="Demucs", label="AI Engine")
            freq_sel = gr.Dropdown(["44100 Hz", "48000 Hz", "96000 Hz"], value="44100 Hz", label="Output Frequency (Sampling Rate)")
            bit_sel = gr.Radio(["16-bit", "24-bit"], value="16-bit", label="Resolution (Bit Depth)")

            run_btn = gr.Button("START PROCESSING", variant="primary")

    gr.Markdown("### 📥 3. Result")
    output_zip = gr.File(label="ZIP Archive (Audio + Report)")

    run_btn.click(process_all, [file_input, yt_input, model_sel, freq_sel, bit_sel], output_zip)

app.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.3/249.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.4 MB/s eta 0:00:00


/tmp/ipykernel_743/1788649361.py:118: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c1e11308b6dcb53c85.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')